# Cross-City Generalization Test

In [1]:
import csv
import os
import random
import sys
import time
from types import ModuleType
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

if "torch.distributed.rpc" not in sys.modules:
    try:
        import torch.distributed.rpc
    except RuntimeError:
        rpc_dummy = ModuleType("torch.distributed.rpc")
        rpc_dummy.is_available = lambda: False
        sys.modules["torch.distributed.rpc"] = rpc_dummy

try:
    import timm
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "timm"])
    import timm
from skimage.filters import threshold_otsu
from sklearn.metrics import cohen_kappa_score
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning, module="timm.layers.interpolate")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/datasets/user/lulc-hyderabad"
OUT_DIR = os.getenv("OUT_DIR", "./outputs")
os.makedirs(OUT_DIR, exist_ok=True)
NUM_CLASSES = 8
CKPT_NAME = "best_model_final_5.pth"
CVA_WATER_NDWI_THRESHOLD = 0.35
DISPLAY_CONFIDENCE_TAU = 0.60

CONFIG = {
    "img_size": 128,
    "in_chans": 9,
    "num_classes": NUM_CLASSES,
    "batch_size": 32,
    "num_workers": 4,
    "lr": 5.0e-5,
    "weight_decay": 6.358359e-4,
    "focal_gamma": 0.6136,
    "backbone_lr_ratio": 0.2719,
    "epochs": 160,
    "freeze_epochs": 15,
    "early_stop_patience": 20,
    "grad_clip_norm": 1.0,
    "label_smoothing": 0.1,
    "class_weight_cap": 5.0,
    "change_pos_weight_cap": 8.0,
    "change_bce_weight": 0.70,
    "change_dice_weight": 0.30,
    "change_task_weight": 1.0,
    "backbone": "swin_tiny_patch4_window7_224",
    "attn_heads": 8,
    "cross_attn_heads": 8,
    "num_layers": 3,
}

print(f"Device: {device}")

Device: cuda


## Spectral Processing

In [2]:
def to_nine_band(patch_5band, eps=1e-6):
    b2, b3, b4, b8, b11 = patch_5band[:5]
    ndvi = (b8 - b4) / (b8 + b4 + eps)
    ndwi = (b3 - b8) / (b3 + b8 + eps)
    ndbi = (b11 - b8) / (b11 + b8 + eps)
    mndwi = (b3 - b11) / (b3 + b11 + eps)
    return np.stack([b2, b3, b4, b8, b11, ndvi, ndwi, ndbi, mndwi], axis=0).astype(np.float32)


def normalize_bands(bands):
    out = bands.copy()
    out[:5] = np.clip(out[:5], 0, 1)
    out[5:] = (np.clip(out[5:], -1, 1) + 1) / 2
    return out

## Model Architecture

In [3]:
class UNetDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch+skip_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),         nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=[1,2,4]):
        super().__init__()
        self.stages = nn.ModuleList([nn.Sequential(
            nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))])
        for r in rates:
            self.stages.append(nn.Sequential(
                nn.Conv2d(in_ch,out_ch,3,padding=r,dilation=r,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True)))
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
        self.project = nn.Sequential(
            nn.Conv2d(out_ch*(len(rates)+2),out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
    def forward(self, x):
        feats = [s(x) for s in self.stages]
        p = F.interpolate(self.pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        feats.append(p)
        return self.project(torch.cat(feats, dim=1))

def widen_patch_embed(backbone, in_chans):
    proj = backbone.patch_embed.proj; old_w = proj.weight.data
    new_proj = nn.Conv2d(in_chans, proj.out_channels, proj.kernel_size, proj.stride, proj.padding,
                         bias=proj.bias is not None)
    with torch.no_grad():
        new_proj.weight[:,:3] = old_w
        if in_chans > 3:
            new_proj.weight[:,3:] = old_w.mean(1,keepdim=True).repeat(1,in_chans-3,1,1)
        if proj.bias is not None: new_proj.bias[:] = proj.bias.data
    backbone.patch_embed.proj = new_proj

def to_nchw(feat, chs):
    if feat.dim()==4 and feat.shape[1]!=feat.shape[-1] and feat.shape[-1] in chs:
        feat = feat.permute(0,3,1,2).contiguous()
    return feat

class BSALayer(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(self, x):
        h=self.n1(x); x=x+self.attn(h,h,h,need_weights=False)[0]
        h=self.n2(x); return x+self.mlp(h)

class BottleneckSelfAttention(nn.Module):
    def __init__(self, dim, heads=8, num_layers=1):
        super().__init__()
        self.layers = nn.ModuleList([BSALayer(dim,heads) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(dim)
    def forward(self, x):
        B,C,H,W = x.shape
        t = x.permute(0,2,3,1).reshape(B,H*W,C)
        for l in self.layers: t=l(t)
        return self.norm(t).reshape(B,H,W,C).permute(0,3,1,2)

class CrossTransformerBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.na=nn.LayerNorm(dim); self.nb=nn.LayerNorm(dim)
        self.attn12=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.attn21=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.nc=nn.LayerNorm(dim); self.nd=nn.LayerNorm(dim)
        self.proj=nn.Linear(dim*2,dim)
    def forward(self, f1, f2):
        B,C,H,W = f1.shape
        t1=f1.permute(0,2,3,1).reshape(B,H*W,C); t2=f2.permute(0,2,3,1).reshape(B,H*W,C)
        n1,n2=self.na(t1),self.nb(t2)
        c1=t1+self.attn12(n1,n2,n2,need_weights=False)[0]
        c2=t2+self.attn21(n2,n1,n1,need_weights=False)[0]
        out=self.proj(torch.cat([self.nc(c1),self.nd(c2)],dim=-1))
        return out.reshape(B,H,W,C).permute(0,3,1,2)

class SiameseSwinRefined(nn.Module):
    def __init__(self, num_classes=8, in_chans=9, img_size=128,
                 backbone_name="swin_tiny_patch4_window7_224",
                 pretrained=True, attn_heads=8, cross_heads=8, num_layers=3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                            features_only=True, img_size=img_size, in_chans=3)
        widen_patch_embed(self.backbone, in_chans)
        self.chs = self.backbone.feature_info.channels()

        self.self_attn   = BottleneckSelfAttention(self.chs[3], attn_heads, num_layers)
        self.aspp        = ASPP(self.chs[3], self.chs[3], rates=[1,2,4])
        self.cross_attn  = CrossTransformerBlock(self.chs[3], cross_heads)

        self.dec3 = UNetDecoderBlock(self.chs[3], self.chs[2], self.chs[2])
        self.dec2 = UNetDecoderBlock(self.chs[2], self.chs[1], self.chs[1])
        self.dec1 = UNetDecoderBlock(self.chs[1], self.chs[0], self.chs[0])

        self.lulc_head = nn.Sequential(
            nn.Conv2d(self.chs[0], self.chs[0]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[0]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[0]//2, num_classes, 1))

        self.change_head = nn.Sequential(
            nn.Conv2d(self.chs[3], self.chs[3]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[3]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[3]//2, 1, 1))

        self.img_size = img_size

    def encode(self, x):
        return [to_nchw(f, self.chs) for f in self.backbone(x)]

    def forward(self, x1, x2):
        f1, f2 = self.encode(x1), self.encode(x2)
        f1a, f2a = self.self_attn(f1[3]), self.self_attn(f2[3])
        f1p, f2p = self.aspp(f1a), self.aspp(f2a)

        s1,s2,s3,_ = f2
        d3 = self.dec3(f2p, s3); d2 = self.dec2(d3,s2); d1 = self.dec1(d2,s1)
        lulc = F.interpolate(self.lulc_head(d1),(self.img_size,self.img_size),mode="bilinear",align_corners=False)

        fused = self.cross_attn(f1p, f2p)
        change = F.interpolate(self.change_head(fused),(self.img_size,self.img_size),mode="bilinear",align_corners=False)
        return lulc, change.squeeze(1)

## Load Checkpoint

In [4]:
model = SiameseSwinRefined(
    num_classes=NUM_CLASSES, in_chans=CONFIG["in_chans"], img_size=CONFIG["img_size"],
    backbone_name=CONFIG["backbone"], pretrained=False,
    attn_heads=CONFIG["attn_heads"], cross_heads=CONFIG["cross_attn_heads"],
    num_layers=CONFIG["num_layers"]
).to(device)

ckpt_path = os.path.join("/kaggle/input/datasets/user/lulc-model-1", CKPT_NAME)
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")


Loaded checkpoint from epoch 158


## TTA

In [5]:
def tta_predict(model, x1, x2):
    lulc_sum = None
    change_sum = None
    for rotation in range(4):
        rotated_x1 = torch.rot90(x1, rotation, (2, 3))
        rotated_x2 = torch.rot90(x2, rotation, (2, 3))
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            lulc_logits, change_logits = model(rotated_x1, rotated_x2)
        lulc_logits = torch.rot90(lulc_logits, -rotation, (2, 3))
        change_logits = torch.rot90(change_logits.unsqueeze(1), -rotation, (2, 3)).squeeze(1)
        lulc_sum = lulc_logits if lulc_sum is None else lulc_sum + lulc_logits
        change_sum = change_logits if change_sum is None else change_sum + change_logits
    return lulc_sum / 4, change_sum / 4

In [6]:
CLASS_NAMES = ["Trees", "Shrubland", "Grassland", "Cropland", "Built-up", "Bare/Sparse", "Water", "Wetland"]

LULC_COLORS = np.array([
    [0, 100, 0], [139, 90, 43], [124, 252, 0], [255, 215, 0],
    [255, 0, 0], [210, 180, 140], [0, 0, 255], [0, 128, 128],
], dtype=np.float32) / 255.0

def label_to_rgb(label_map, colors=LULC_COLORS):
    rgb = np.zeros((*label_map.shape, 3), dtype=np.float32)
    for class_index in range(len(colors)):
        rgb[label_map == class_index] = colors[class_index]
    return rgb

## Load Cross-City Data


In [7]:
# Select city: 'Hyderabad', 'Bengaluru', or 'Pune'
CROSS_CITY_NAME = "Hyderabad"

city_prefix = CROSS_CITY_NAME.lower()
p_t1 = os.path.join(DATA_DIR, f"{city_prefix}_patches_2019.npy")
p_t2 = os.path.join(DATA_DIR, f"{city_prefix}_patches_2024.npy")
p_single = os.path.join(DATA_DIR, f"{city_prefix}_patches.npy")

if os.path.exists(p_t1) and os.path.exists(p_t2):
    cross_patches_t1 = np.load(p_t1)
    cross_patches_t2 = np.load(p_t2)
    is_bitemporal = True
    n_cross = cross_patches_t1.shape[0]
    print(f"Loaded {CROSS_CITY_NAME} bi-temporal datasets: 2019 ({cross_patches_t1.shape}) and 2024 ({cross_patches_t2.shape})")
elif os.path.exists(p_single):
    cross_patches_t1 = np.load(p_single)
    cross_patches_t2 = cross_patches_t1
    is_bitemporal = False
    n_cross = cross_patches_t1.shape[0]
    print(f"Loaded {CROSS_CITY_NAME} dataset: shape={cross_patches_t1.shape} (single year)")
else:
    raise FileNotFoundError(f"Could not find {CROSS_CITY_NAME} patch files in {DATA_DIR}")


Loaded Hyderabad bi-temporal datasets: 2019 ((64, 5, 128, 128)) and 2024 ((64, 5, 128, 128))


## Run Cross-City Inference

In [8]:
cross_preds_t1 = np.zeros((n_cross, CONFIG["img_size"], CONFIG["img_size"]), dtype=np.uint8)
cross_preds_t2 = np.zeros((n_cross, CONFIG["img_size"], CONFIG["img_size"]), dtype=np.uint8)
cross_change_probs = np.zeros((n_cross, CONFIG["img_size"], CONFIG["img_size"]), dtype=np.float32)

print(f"Running Siamese Swin-T cross-city inference on {n_cross} {CROSS_CITY_NAME} patches...")
batch_size = 8
for i in range(0, n_cross, batch_size):
    end_idx = min(i + batch_size, n_cross)
    b1 = torch.from_numpy(np.stack([normalize_bands(to_nine_band(cross_patches_t1[j])) for j in range(i, end_idx)])).to(device)
    b2 = torch.from_numpy(np.stack([normalize_bands(to_nine_band(cross_patches_t2[j])) for j in range(i, end_idx)])).to(device)

    with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
        lulc1, _ = model(b1, b1)
        lulc2, chg = model(b1, b2)

    cross_preds_t1[i:end_idx] = lulc1.argmax(1).cpu().numpy().astype(np.uint8)
    cross_preds_t2[i:end_idx] = lulc2.argmax(1).cpu().numpy().astype(np.uint8)
    cross_change_probs[i:end_idx] = torch.sigmoid(chg).cpu().numpy()

print(f"Inference complete for all {n_cross} patches.")


Running Siamese Swin-T cross-city inference on 64 Hyderabad patches...
Inference complete for all 64 patches.


## Visualize Cross-City Results

In [9]:
def make_display_rgb(img_5band):
    nine = normalize_bands(to_nine_band(img_5band))
    rgb = np.stack([nine[2], nine[1], nine[0]], axis=-1)
    lo, hi = np.percentile(rgb, 2), np.percentile(rgb, 98)
    return np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)

n_show = min(4, n_cross)
if is_bitemporal:
    fig, axes = plt.subplots(n_show, 5, figsize=(22, 4.5 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]
    for i in range(n_show):
        axes[i, 0].imshow(make_display_rgb(cross_patches_t1[i]))
        axes[i, 0].set_title(f"Patch {i}: S2 RGB (2019)", fontweight="bold")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(label_to_rgb(cross_preds_t1[i]))
        axes[i, 1].set_title("Predicted LULC (2019)", fontweight="bold")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(make_display_rgb(cross_patches_t2[i]))
        axes[i, 2].set_title(f"Patch {i}: S2 RGB (2024)", fontweight="bold")
        axes[i, 2].axis("off")

        axes[i, 3].imshow(label_to_rgb(cross_preds_t2[i]))
        axes[i, 3].set_title("Predicted LULC (2024)", fontweight="bold")
        axes[i, 3].axis("off")

        im = axes[i, 4].imshow(cross_change_probs[i], cmap="magma", vmin=0, vmax=1)
        axes[i, 4].set_title(f"Change Prob (Mean: {cross_change_probs[i].mean():.2f})", fontweight="bold")
        axes[i, 4].axis("off")
    fig.suptitle(f"Zero-Shot Bi-Temporal Generalization: {CROSS_CITY_NAME} (2019 vs 2024)", fontsize=16, fontweight="bold")
else:
    fig, axes = plt.subplots(n_show, 3, figsize=(18, 5 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]
    for i in range(n_show):
        axes[i, 0].imshow(make_display_rgb(cross_patches_t1[i]))
        axes[i, 0].set_title(f"Patch {i}: S2 RGB", fontweight="bold")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(label_to_rgb(cross_preds_t1[i]))
        axes[i, 1].set_title("Predicted LULC", fontweight="bold")
        axes[i, 1].axis("off")

        pcts = [(cross_preds_t1[i] == c).mean() * 100 for c in range(NUM_CLASSES)]
        axes[i, 2].barh(CLASS_NAMES, pcts, color=[LULC_COLORS[c] for c in range(NUM_CLASSES)], edgecolor="black", linewidth=0.5)
        axes[i, 2].set_xlim(0, 100)
        axes[i, 2].set_title("Class Distribution (%)", fontweight="bold")
        axes[i, 2].grid(axis="x", alpha=0.3)
    fig.suptitle(f"Cross-City Generalization Test — {CROSS_CITY_NAME}", fontsize=16, fontweight="bold")

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "cross_city_results.png"), dpi=180, bbox_inches="tight")
plt.show()
print("Saved cross_city_results.png")


Saved cross_city_results.png


## Cross-City Class Distribution Summary

In [10]:
flat_t1 = cross_preds_t1.reshape(-1)
flat_t2 = cross_preds_t2.reshape(-1)
total_pix = len(flat_t1)

print(f"\n{CROSS_CITY_NAME} Class Distribution Summary (Total Pixels: {total_pix:,d}):")
if is_bitemporal:
    print(f"{'Class':>15s} | {'2019 %':>9s} | {'2024 %':>9s} | {'Net Change':>12s}")
    print("-" * 53)
    for c in range(NUM_CLASSES):
        pct1 = (flat_t1 == c).sum() / total_pix * 100
        pct2 = (flat_t2 == c).sum() / total_pix * 100
        diff = pct2 - pct1
        print(f"{CLASS_NAMES[c]:>15s} | {pct1:>8.2f}% | {pct2:>8.2f}% | {diff:>+11.2f}%")
    chg_rate = (cross_change_probs > 0.5).mean() * 100
    print(f"\nMean Bi-Temporal Change Detection Rate (Prob > 0.5): {chg_rate:.2f}%")
else:
    print(f"{'Class':>15s} | {'Pixels':>12s} | {'%':>8s}")
    print("-" * 40)
    for c in range(NUM_CLASSES):
        cnt = (flat_t1 == c).sum()
        pct = cnt / total_pix * 100
        print(f"{CLASS_NAMES[c]:>15s} | {cnt:>12,d} | {pct:>7.2f}%")



Hyderabad Class Distribution Summary (Total Pixels: 1,048,576):
          Class |    2019 % |    2024 % |   Net Change
-----------------------------------------------------
          Trees |    14.97% |    19.89% |       +4.92%
      Shrubland |     0.81% |     0.09% |       -0.73%
      Grassland |     3.63% |     1.10% |       -2.53%
       Cropland |     2.59% |     0.74% |       -1.86%
       Built-up |    72.51% |    73.53% |       +1.02%
    Bare/Sparse |     1.50% |     0.72% |       -0.78%
          Water |     3.85% |     3.71% |       -0.14%
        Wetland |     0.14% |     0.23% |       +0.09%

Mean Bi-Temporal Change Detection Rate (Prob > 0.5): 33.49%


## Continuous Cross-City Mosaic Visualization


In [11]:
GRID_N = int(np.sqrt(n_cross))
canvas_t1_rgb = np.zeros((GRID_N * 128, GRID_N * 128, 3), dtype=np.float32)
canvas_t1_lulc = np.zeros((GRID_N * 128, GRID_N * 128), dtype=np.uint8)
canvas_t2_rgb = np.zeros((GRID_N * 128, GRID_N * 128, 3), dtype=np.float32)
canvas_t2_lulc = np.zeros((GRID_N * 128, GRID_N * 128), dtype=np.uint8)
canvas_change = np.zeros((GRID_N * 128, GRID_N * 128), dtype=np.float32)

for idx in range(n_cross):
    r = idx // GRID_N
    c = idx % GRID_N
    y, x = r * 128, c * 128
    canvas_t1_rgb[y:y+128, x:x+128] = make_display_rgb(cross_patches_t1[idx])
    canvas_t1_lulc[y:y+128, x:x+128] = cross_preds_t1[idx]
    canvas_t2_rgb[y:y+128, x:x+128] = make_display_rgb(cross_patches_t2[idx])
    canvas_t2_lulc[y:y+128, x:x+128] = cross_preds_t2[idx]
    canvas_change[y:y+128, x:x+128] = cross_change_probs[idx]

if is_bitemporal:
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    axes[0].imshow(label_to_rgb(canvas_t1_lulc))
    axes[0].set_title(f"{CROSS_CITY_NAME} 2019 — Predicted LULC", fontsize=13, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(label_to_rgb(canvas_t2_lulc))
    axes[1].set_title(f"{CROSS_CITY_NAME} 2024 — Predicted LULC", fontsize=13, fontweight="bold")
    axes[1].axis("off")

    im = axes[2].imshow(canvas_change, cmap="magma", vmin=0, vmax=1)
    axes[2].set_title(f"{CROSS_CITY_NAME} (2019-2024) — Change Probability Mosaic", fontsize=13, fontweight="bold")
    axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04, label="Change Probability")
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    axes[0].imshow(canvas_t1_rgb)
    axes[0].set_title(f"{CROSS_CITY_NAME} — Sentinel-2 Natural Color RGB (10.24x10.24 km)", fontsize=13, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(label_to_rgb(canvas_t1_lulc))
    axes[1].set_title(f"{CROSS_CITY_NAME} — Zero-Shot Predicted LULC", fontsize=13, fontweight="bold")
    axes[1].axis("off")

legend_patches = [mpatches.Patch(color=LULC_COLORS[c], label=CLASS_NAMES[c]) for c in range(NUM_CLASSES)]
fig.legend(handles=legend_patches, loc="lower center", ncol=4, fontsize=11, frameon=True,
           title="Predicted Land Cover Class", bbox_to_anchor=(0.5, -0.05))
fig.suptitle(f"Zero-Shot Cross-City Generalization: {CROSS_CITY_NAME} Central Region (~105 km²)", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "cross_city_mosaic.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved cross_city_mosaic.png")


Saved cross_city_mosaic.png
